Install Dependencies

In [ ]:
!pip install groq --quiet

In [ ]:
import os
from groq import Groq

Setup API

In [ ]:
GROQ_API_KEY = "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"
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [ ]:
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [ ]:
conversation_history = []

def add_message(role, content):
    conversation_history.append({"role": role, "content": content})

In [ ]:
def truncate_history_by_turns(history, max_turns):
    return history[-max_turns:]

In [ ]:
def truncate_history_by_chars(history, max_chars):
    combined = ""
    truncated = []
    for message in reversed(history):
        if len(combined) + len(message["content"]) > max_chars:
            break
        combined = message["content"] + combined
        truncated.insert(0, message)
    return truncated

In [ ]:
def summarize_history(history):
    text = "\n".join([f'{m["role"]}: {m["content"]}' for m in history])

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are an assistant that summarizes conversations clearly and briefly."},
            {"role": "user", "content": f"Summarize this conversation:\n{text}"}
        ],
    )
    return response.choices[0].message.content

In [ ]:
def periodic_summarization(history, run_count, k):
    if run_count > 0 and run_count % k == 0:
        summary = summarize_history(history)
        print(f"\n*** Periodic Summarization after {run_count} runs ***")
        print(summary)
        return [{"role": "assistant", "content": summary}]
    return history

In [ ]:
def run_conversation():
    run_count = 0
    max_turns_to_keep = 10
    max_chars_to_keep = 2000
    summarization_frequency = 3

    print("Start chatting with the assistant (type 'exit' to quit):\n")

    while True:
        user_input = input("You: ")
        if user_input.lower() in ["exit", "quit"]:
            print("Exiting chat.")
            break

        add_message("user", user_input)

        truncated_history = truncate_history_by_turns(conversation_history, max_turns_to_keep)
        # Alternatively: truncated_history = truncate_history_by_chars(conversation_history, max_chars_to_keep)

        current_history = periodic_summarization(truncated_history, run_count, summarization_frequency)

        api_messages = current_history + [{"role": "user", "content": user_input}]

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=api_messages,
        )

        assistant_reply = response.choices[0].message.content
        print()
        print("Assistant:", assistant_reply)
        add_message("assistant", assistant_reply)
        print()

        run_count += 1

In [ ]:


run_conversation()

Start chatting with the assistant (type 'exit' to quit):

You: hi there who are you?

Assistant: Hello. I'm an artificial intelligence model known as a chatbot or conversational AI. I'm a computer program designed to simulate conversation, answer questions, and provide information on a wide range of topics. I'm here to help you with any questions or topics you'd like to discuss. What would you like to talk about?

You: lets talk about laliga

Assistant: LaLiga is the top professional football division in Spain, known for its passionate fans, tactical football, and talented players. It's home to legendary clubs like Barcelona, Real Madrid, and Atlético Madrid, among others.

What aspect of LaLiga would you like to discuss?

- Recent season highlights?
- Club rivalries (El Clásico, etc.)?
- Spanish national team prospects?
- A specific team or player's performance?
- Something else?

Let me know, and I'll do my best to engage in the conversation!

You: who do you think will win it this t